# Type-Based Baseline Experiment

**Goal**: Compare coverage with a simple type-based baseline.

If type-baseline performs similarly to coverage, it means coverage is just detecting type violations.
If coverage outperforms type-baseline, it captures deeper observation patterns.

**Memory**: ~2GB GPU (lightweight, mostly CPU)

In [1]:
import numpy as np
from sklearn.metrics import roc_auc_score
from collections import defaultdict
import json
import os
import urllib.request
import random

print("Type-Based Baseline Experiment")
print("This runs mostly on CPU, minimal GPU needed.")

Type-Based Baseline Experiment
This runs mostly on CPU, minimal GPU needed.


In [2]:
# Download FB15k-237
def download_fb15k237():
    os.makedirs('data', exist_ok=True)
    base_url = "https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/FB15k-237"
    for split in ['train', 'test']:
        path = f'data/{split}.txt'
        if not os.path.exists(path):
            print(f"Downloading {split}...")
            urllib.request.urlretrieve(f"{base_url}/{split}.txt", path)
    print("Download complete!")

download_fb15k237()

def load_triples(path):
    triples = []
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                triples.append((parts[0], parts[1], parts[2]))
    return triples

train = load_triples('data/train.txt')
test = load_triples('data/test.txt')

entities = set()
relations = set()
for h, r, t in train + test:
    entities.add(h)
    entities.add(t)
    relations.add(r)

ent2idx = {e: i for i, e in enumerate(entities)}
rel2idx = {r: i for i, r in enumerate(relations)}

print(f"Entities: {len(entities)}, Relations: {len(relations)}")

Download complete!
Entities: 14534, Relations: 237


## 1. Build Type Information

For FB15k-237, we infer types from relation domain/range.

In [3]:
def build_relation_types(triples):
    """
    Build relation domain/range types.

    For each relation, track which entities appear as head (domain) and tail (range).
    """
    relation_heads = defaultdict(set)  # relation -> valid head entities
    relation_tails = defaultdict(set)  # relation -> valid tail entities

    for h, r, t in triples:
        relation_heads[r].add(h)
        relation_tails[r].add(t)

    print(f"Built type constraints for {len(relation_heads)} relations")

    # Stats
    avg_heads = np.mean([len(v) for v in relation_heads.values()])
    avg_tails = np.mean([len(v) for v in relation_tails.values()])
    print(f"Avg valid heads per relation: {avg_heads:.1f}")
    print(f"Avg valid tails per relation: {avg_tails:.1f}")

    return relation_heads, relation_tails

relation_heads, relation_tails = build_relation_types(train)

Built type constraints for 237 relations
Avg valid heads per relation: 394.0
Avg valid tails per relation: 237.6


## 2. Type-Based Baseline

Uncertainty = whether entity is in the valid domain/range of the relation.

In [4]:
class TypeBaseline:
    """Type-based OOD detector: flag if entity not in relation's domain/range."""

    def __init__(self, relation_heads, relation_tails):
        self.relation_heads = relation_heads
        self.relation_tails = relation_tails

    def get_uncertainty(self, triples):
        """
        Return uncertainty for each triple.

        0 = both entities in valid domain/range
        1 = one entity invalid
        2 = both entities invalid
        """
        uncertainties = []
        for h, r, t in triples:
            h_valid = h in self.relation_heads.get(r, set())
            t_valid = t in self.relation_tails.get(r, set())
            unc = 2 - int(h_valid) - int(t_valid)
            uncertainties.append(unc)
        return np.array(uncertainties)


class CoverageBaseline:
    """Coverage-based OOD detector: flag if entity-relation pair not observed."""

    def __init__(self):
        self.coverage = defaultdict(lambda: defaultdict(bool))

    def fit(self, triples):
        for h, r, t in triples:
            self.coverage[h][r] = True
            self.coverage[t][r] = True
        return self

    def get_uncertainty(self, triples):
        uncertainties = []
        for h, r, t in triples:
            h_cov = self.coverage[h].get(r, False)
            t_cov = self.coverage[t].get(r, False)
            unc = 2 - int(h_cov) - int(t_cov)
            uncertainties.append(unc)
        return np.array(uncertainties)

## 3. Evaluation

In [5]:
def evaluate_auroc(method, test_triples, ood_triples):
    """Compute AUROC for OOD detection."""
    id_unc = method.get_uncertainty(test_triples)
    ood_unc = method.get_uncertainty(ood_triples)

    labels = np.concatenate([np.zeros(len(id_unc)), np.ones(len(ood_unc))])
    scores = np.concatenate([id_unc, ood_unc])

    return roc_auc_score(labels, scores)


def generate_random_ood(test_triples, all_entities):
    """Generate OOD by random tail corruption."""
    ood = []
    for h, r, t in test_triples:
        t_ood = random.choice(list(all_entities))
        ood.append((h, r, t_ood))
    return ood

## 4. Main Comparison

In [6]:
seeds = [42, 123, 456]
results = {'type_baseline': [], 'coverage': []}

# Initialize methods
type_baseline = TypeBaseline(relation_heads, relation_tails)
coverage = CoverageBaseline().fit(train)

for seed in seeds:
    print(f"\nSeed {seed}")
    random.seed(seed)
    np.random.seed(seed)

    # Generate OOD
    ood = generate_random_ood(test, entities)

    # Evaluate
    type_auroc = evaluate_auroc(type_baseline, test, ood)
    cov_auroc = evaluate_auroc(coverage, test, ood)

    results['type_baseline'].append(type_auroc)
    results['coverage'].append(cov_auroc)

    print(f"  Type-baseline: {type_auroc:.4f}")
    print(f"  Coverage:      {cov_auroc:.4f}")


Seed 42
  Type-baseline: 0.8521
  Coverage:      0.8200

Seed 123
  Type-baseline: 0.8525
  Coverage:      0.8206

Seed 456
  Type-baseline: 0.8526
  Coverage:      0.8212


In [7]:
print("\n" + "="*60)
print("RESULTS: FB15k-237")
print("="*60)

type_mean = np.mean(results['type_baseline'])
type_std = np.std(results['type_baseline'])
cov_mean = np.mean(results['coverage'])
cov_std = np.std(results['coverage'])

print(f"\nType-baseline: {type_mean:.4f} ± {type_std:.4f}")
print(f"Coverage:      {cov_mean:.4f} ± {cov_std:.4f}")

print("\n--- Analysis ---")
gap = cov_mean - type_mean
if gap > 0.01:
    print(f"✓ Coverage outperforms type-baseline by {gap:.4f} ({gap/type_mean*100:.1f}%)")
    print("  Coverage captures more than just type violations.")
    print("  It detects observation patterns WITHIN valid types.")
elif gap < -0.01:
    print(f"✗ Type-baseline outperforms coverage by {-gap:.4f}")
    print("  Coverage may be partially redundant with type information.")
else:
    print(f"≈ Coverage and type-baseline perform similarly (gap: {gap:.4f})")
    print("  Coverage may be capturing mostly type information.")


RESULTS: FB15k-237

Type-baseline: 0.8524 ± 0.0002
Coverage:      0.8206 ± 0.0005

--- Analysis ---
✗ Type-baseline outperforms coverage by 0.0318
  Coverage may be partially redundant with type information.


In [8]:
# Analyze where they differ
print("\n--- Detailed Analysis ---")

random.seed(42)
ood = generate_random_ood(test, entities)

type_unc = type_baseline.get_uncertainty(test + ood)
cov_unc = coverage.get_uncertainty(test + ood)

# Cases where they differ
n = len(test)
id_type = type_unc[:n]
id_cov = cov_unc[:n]
ood_type = type_unc[n:]
ood_cov = cov_unc[n:]

# For ID: coverage uncertain but type confident
id_cov_warns_type_doesnt = np.sum((id_cov > 0) & (id_type == 0))
print(f"ID triples where coverage is uncertain but type is confident: {id_cov_warns_type_doesnt} ({id_cov_warns_type_doesnt/n*100:.1f}%)")

# For OOD: coverage catches but type doesn't
ood_cov_catches_type_doesnt = np.sum((ood_cov > 0) & (ood_type == 0))
print(f"OOD triples where coverage catches but type doesn't: {ood_cov_catches_type_doesnt} ({ood_cov_catches_type_doesnt/n*100:.1f}%)")

# Vice versa
ood_type_catches_cov_doesnt = np.sum((ood_type > 0) & (ood_cov == 0))
print(f"OOD triples where type catches but coverage doesn't: {ood_type_catches_cov_doesnt} ({ood_type_catches_cov_doesnt/n*100:.1f}%)")


--- Detailed Analysis ---
ID triples where coverage is uncertain but type is confident: 0 (0.0%)
OOD triples where coverage catches but type doesn't: 0 (0.0%)
OOD triples where type catches but coverage doesn't: 1197 (5.8%)


In [9]:
# Save results
output = {
    'dataset': 'FB15k-237',
    'results': {
        'type_baseline': {'mean': float(type_mean), 'std': float(type_std)},
        'coverage': {'mean': float(cov_mean), 'std': float(cov_std)},
    },
    'analysis': {
        'coverage_advantage': float(gap),
        'ood_cov_catches_type_misses_pct': float(ood_cov_catches_type_doesnt/n*100),
    }
}

with open('type_baseline_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("\nResults saved to type_baseline_results.json")


Results saved to type_baseline_results.json
